# Linear regression: a tiny ordinary-least-squares lab

Companion to HTB Academy Module 290, Section 4. HTB explains the model and OLS conceptually; this local notebook adds a small, deterministic demonstration. The data are invented and are **not** a real security detector or a challenge artifact.

Read $\hat y=mx+c$ aloud as *y-hat equals m times x plus c*. The hat says prediction; $m$ is slope; $c$ is intercept. Read $r_i=y_i-\hat y_i$ as *r sub i equals y sub i minus y-hat sub i*. A residual is one row's signed error.

In [ ]:
import numpy as np

# Three synthetic observations: request-size units (x) and latency units (y).
# Their small size lets us inspect every prediction and residual by hand.
x = np.array([1.0, 2.0, 3.0])
y = np.array([2.0, 3.0, 5.0])
assert x.shape == y.shape == (3,)

## Score one candidate line

Read $RSS=\sum_{i=1}^{N}(y_i-\hat y_i)^2$ as *R-S-S equals the sum from i equals one to N of y sub i minus y-hat sub i, squared*. For each row, subtract the prediction from the observation, square, then add. The square stops positive and negative errors from cancelling.

In [ ]:
# Try the hand-chosen line: y-hat = 1*x + 1.
candidate_slope, candidate_intercept = 1.0, 1.0
candidate_predictions = candidate_slope * x + candidate_intercept
candidate_residuals = y - candidate_predictions  # Formula: r_i = y_i - y-hat_i.
candidate_rss = np.sum(candidate_residuals ** 2)  # Formula: RSS = sum_i r_i^2.
print('candidate predictions:', candidate_predictions)
print('candidate residuals:', candidate_residuals)
print('candidate RSS:', candidate_rss)
assert np.allclose(candidate_residuals, [0, 0, 1])
assert np.isclose(candidate_rss, 1.0)

## Fit the least-squares line

Read $\operatorname*{arg\,min}_{m,c} RSS(m,c)$ as *the values of m and c that minimize R-S-S*. `arg min` returns the coefficient choices, rather than the score. The two-column design matrix below has one column of input values and one of ones; multiplying it by `[slope, intercept]` produces $mx+c$ for every row. `np.linalg.lstsq` is a numerical implementation of this course concept, not a course-provided snippet.

In [ ]:
# Column 0 supplies x_i; column 1 supplies a constant 1 for the intercept.
design = np.column_stack([x, np.ones_like(x)])
assert design.shape == (3, 2)  # 3 observations, 2 coefficients.

# Least squares chooses slope and intercept to minimize RSS.
(slope, intercept), _, _, _ = np.linalg.lstsq(design, y, rcond=None)
fitted_predictions = design @ np.array([slope, intercept])  # Formula: y-hat_i = m*x_i + c.
fitted_residuals = y - fitted_predictions
fitted_rss = np.sum(fitted_residuals ** 2)

print(f'slope={slope:.6f}, intercept={intercept:.6f}')
print('fitted predictions:', fitted_predictions)
print('fitted residuals:', fitted_residuals)
print('fitted RSS:', fitted_rss)
assert np.isclose(slope, 1.5)
assert np.isclose(intercept, 1 / 3)
assert np.isclose(fitted_rss, 1 / 6)
assert fitted_rss < candidate_rss

The fitted line has a smaller *training* RSS than our hand-chosen line. That comparison alone says nothing about future requests. A real model also needs held-out evaluation and assumption checks; this three-row exercise exists only to expose the arithmetic.